In [1]:
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import ssl
import certifi
import random
from datetime import datetime

import os
from dotenv import load_dotenv
from weaviate.classes.config import Property, DataType
import os
import uuid
import base64
import httpx
from datetime import datetime
from typing import List, Dict, Any, Optional, Annotated, Literal, TypedDict
import sys
sys.path.insert(0, r'..\src')

In [2]:
load_dotenv(os.path.join("..", ".env"), override=True)

%load_ext autoreload
%autoreload 2

def summarize_value(value: str) -> str:
    """Return masked form: ****last4 or boolean string."""
    lower = value.lower()
    if lower in ("true", "false"):
        return lower
    return "****" + value[-4:] if len(value) > 4 else "****" + value

def doublecheck_env(file_path: str):
    """Check environment variables against a .env file and print summaries."""
    if not os.path.exists(file_path):
        print(f"Did not find file {file_path}.")
        print("This is used to double check the key settings for the notebook.")
        print("This is just a check and is not required.\n")
        return

    parsed = dotenv_values(file_path)
    for key in parsed.keys():
        current = os.getenv(key)
        if current is not None:
            print(f"{key}={summarize_value(current)}")
        else:
            print(f"{key}=<not set>")
            

load_dotenv(os.path.join("..", ".env"), override=True)

In [4]:
from utils import show_prompt
from prompts_arxiv import SUMMARIZE_WEB_SEARCH
show_prompt(SUMMARIZE_WEB_SEARCH)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│  You are creating a minimal summary for research steering - your goal is to help an agent know what             │
│  information it has collected, NOT to preserve all details.                                                     │
│                                                                                                                 │
│  <webpage_content>                                                                                              │
│  {webpage_content}                                                                                              │
│  </webpage_content>                                                                                             │
│                                                                                                                 │
│  Create a VERY CONCISE summary focusing on:                                                                     │
│  1. Main topic/subject in 1-2 sentences                                                                         │
│  2. Key information type (facts, tutorial, news, analysis, etc.)                                                │
│  3. Most significant 1-2 findings or points                                                                     │
│                                                                                                                 │
│  Keep the summary under 150 words total. The agent needs to know what's in this file to decide if it should     │
│  search for more information or use this source.                                                                │
│                                                                                                                 │
│  Generate a descriptive filename that indicates the content type and topic (e.g., "mcp_protocol_overview.md",   │
│  "ai_safety_research_2024.md").                                                                                 │
│                                                                                                                 │
│  Output format:                                                                                                 │
│  ```json                                                                                                        │
│  {{                                                                                                             │
│     "filename": "descriptive_filename.md",                                                                      │
│     "summary": "Very brief summary under 150 words focusing on main topic and key findings"                     │
│  }}                                                                                                             │
│  ```                                                                                                            │
│                                                                                                                 │
│  Today's date: {date}                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

%%writefile ../src/research_tools_arxiv.py

"""Research Tools.

This module provides search and content processing utilities for the research agent,
including web search capabilities and content summarization tools.
"""

# -------------------- Imports --------------------
import weaviate
import weaviate.classes as wvc
from sentence_transformers import SentenceTransformer
from langchain_core.messages import ToolMessage, HumanMessage
from langchain_core.tools import InjectedToolCallId, tool
from langchain_core.documents import Document
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
from markdownify import markdownify
from tavily import TavilyClient
from langchain_core.messages import AnyMessage, BaseMessage
from langgraph.graph.message import add_messages
#from prompts import SUMMARIZE_WEB_SEARCH
#from prompts import WRITE_TODOS_DESCRIPTION

#
from task_tool_arxiv import _create_task_tool
from state_arxiv import DeepAgentState



import os
from datetime import datetime
import uuid, base64

import httpx
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import InjectedToolArg, InjectedToolCallId, tool
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from markdownify import markdownify
from pydantic import BaseModel, Field
from tavily import TavilyClient
from typing_extensions import Annotated, Literal, List

from prompts_arxiv import SUMMARIZE_WEB_SEARCH
from state_arxiv import DeepAgentState

# Summarization model 
summarization_model = init_chat_model(model="openai:gpt-4o-mini")
tavily_client = TavilyClient()

class Summary(BaseModel):
    filename: str = Field(description="Name of the file to store.")
    summary: str = Field(description="Key learnings from the webpage.")

def get_today_str() -> str:
    return datetime.now().strftime("%a %b %d, %Y").replace(" 0", " ")

def run_tavily_search(
    search_query: str, 
    max_results: int = 1, 
    include_raw_content: bool = True, 
) -> dict:
    """Perform search using Tavily API for a single query.

    Args:
        search_query: Search query to execute
        max_results: Maximum number of results per query
        include_raw_content: Whether to include raw webpage content

    Returns:
        Search results dictionary
    """
    result = tavily_client.search(
        search_query,
        max_results=max_results,
        include_raw_content=include_raw_content
    )
    return result

def summarize_webpage_content(webpage_content: str) -> Summary:
    """Summarize webpage content using the configured summarization model.

    Args:
        webpage_content: Raw webpage content to summarize

    Returns:
        Summary object with filename and summary
    """
    try:
        structured_model = summarization_model.with_structured_output(Summary)
        summary_and_filename = structured_model.invoke([
            HumanMessage(content=SUMMARIZE_WEB_SEARCH.format(
                webpage_content=webpage_content, 
                date=get_today_str()
            ))
        ])
        return summary_and_filename
    except Exception:
        return Summary(
            filename="search_result.md",
            summary=webpage_content[:1000] + "..." if len(webpage_content) > 1000 else webpage_content
        )

def process_search_results(results: dict) -> List[dict]:
    """Process search results by summarizing content where available.

    Args:
        results: Tavily search results dictionary

    Returns:
        List of processed results with summaries
    """
    HTTPX_CLIENT = httpx.Client(timeout=30.0)
    processed_results = []
    for result in results.get('results', []):
        url = result['url']
        try:
            response = HTTPX_CLIENT.get(url)
            if response.status_code == 200:
                raw_content = markdownify(response.text)
                summary_obj = summarize_webpage_content(raw_content)
            else:
                raw_content = result.get('raw_content', '')
                summary_obj = Summary(
                    filename="URL_error.md",
                    summary=result.get('content', 'Error reading URL; try another search.')
                )
        except (httpx.TimeoutException, httpx.RequestError):
            raw_content = result.get('raw_content', '')
            summary_obj = Summary(
                filename="connection_error.md",
                summary=result.get('content', 'Could not fetch URL (timeout/connection error). Try another search.')
            )
        # uniquify file names
        uid = base64.urlsafe_b64encode(uuid.uuid4().bytes).rstrip(b"=").decode("ascii")[:8]
        name, ext = os.path.splitext(summary_obj.filename)
        summary_obj.filename = f"{name}_{uid}{ext}"
        processed_results.append({
            'url': url,
            'title': result['title'],
            'summary': summary_obj.summary,
            'filename': summary_obj.filename,
            'raw_content': raw_content,
        })
    return processed_results


@tool(parse_docstring=True)
def tavily_search(
    query: str,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
    max_results: int = 1
) -> Command:
    """Search the web and save detailed results to files while returning a minimal context summary.

    Performs a Tavily search, fetches full webpage content for each result,
    summarizes it, saves the raw content and summary to files in the virtual
    filesystem, and returns a brief overview to the agent.

    Args:
        query: The search query string.
        state: Injected agent state containing the current file system.
        tool_call_id: Injected tool call identifier for message responses.
        max_results: Maximum number of search results to return (default: 1)

    Returns:
        Command updating the agent's files with saved search results and
        adding a tool message containing a summary of what was saved.
    """
    search_results = run_tavily_search(query, max_results=max_results, include_raw_content=True)
    processed_results = process_search_results(search_results)
    files = state.get("files", {})
    saved_files = []
    summaries = []
    for result in processed_results:
        filename = result['filename']
        file_content = f"""# Search Result: {result['title']}

                **URL:** {result['url']}
                **Query:** {query}
                **Date:** {get_today_str()}
                
                ## Summary
                {result['summary']}
                
                ## Raw Content
                {result['raw_content'] if result['raw_content'] else 'No raw content available'}
                """
        files[filename] = file_content
        saved_files.append(filename)
        summaries.append(f"- {filename}: {result['summary']}...")
    summary_text = f"""🔍 Found {len(processed_results)} result(s) for '{query}':

{chr(10).join(summaries)}

Files: {', '.join(saved_files)}

💡 Use read_file() to access full details when needed."""
    return Command(
        update={
            "files": files,
            "messages": [
                ToolMessage(summary_text, tool_call_id=tool_call_id)
            ],
        }
    )

@tool(parse_docstring=True)
def think_tool(reflection: str) -> str:
    """Tool for strategic reflection on research progress and decision-making.

    Use this tool after each search to analyze results and plan next steps systematically.
    This creates a deliberate pause in the research workflow for quality decision-making.

    When to use:
    - After receiving search results: What key information did I find?
    - Before deciding next steps: Do I have enough to answer comprehensively?
    - When assessing research gaps: What specific information am I still missing?
    - Before concluding research: Can I provide a complete answer now?
    - How complex is the question: Have I reached the number of search limits?

    Reflection should address:
    1. Analysis of current findings - What concrete information have I gathered?
    2. Gap assessment - What crucial information is still missing?
    3. Quality evaluation - Do I have sufficient evidence/examples for a good answer?
    4. Strategic decision - Should I continue searching or provide my answer?

    Args:
        reflection: Your detailed reflection on research progress, findings, gaps, and next steps

    Returns:
        Confirmation that reflection was recorded for decision-making
    """
    return f"Reflection recorded: {reflection}"

In [5]:
from datetime import datetime
from sentence_transformers import SentenceTransformer
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
#from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from utils import show_prompt, stream_agent

from file_tools_arxiv import ls, read_file, write_file
from prompts_arxiv import (
    FILE_USAGE_INSTRUCTIONS,
    RESEARCHER_INSTRUCTIONS,
    SUBAGENT_USAGE_INSTRUCTIONS,
    TODO_USAGE_INSTRUCTIONS,
)
from research_tools_arxiv import tavily_search, think_tool, get_today_str
from state_arxiv import DeepAgentState
from task_tool_arxiv import _create_task_tool
from todo_tools_arxiv import write_todos, read_todos

import weaviate
import weaviate.classes as wvc
from sentence_transformers import SentenceTransformer
from langchain_core.messages import ToolMessage, HumanMessage
from langchain_core.tools import InjectedToolCallId, tool
from langchain_core.documents import Document
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
from markdownify import markdownify
from tavily import TavilyClient
from langchain_core.messages import AnyMessage, BaseMessage
from langgraph.graph.message import add_messages
BASE_URL = "https://export.arxiv.org/api/query"

def get_arxiv_papers(
    query,
    max_results=3,
    start=0,
    mode=None,          # None → default relevance
    random_pool=100
):
    """
    Retrieve arXiv papers with extended metadata.

    Parameters:
        query (str): search keywords
        max_results (int): number of results to return
        start (int): pagination offset
        mode (str|None): 'latest', 'updated', 'random', or None (relevance)
        random_pool (int): pool size for random mode (used only for random)

    Returns:
        List[dict]: each dict contains paper metadata
    """
    import time
    time.sleep(3)
    if not query:
        raise ValueError("query must not be empty")

    search_query = query

    # Sorting logic
    if mode is None:
        sort_by = "relevance"
        sort_order = "descending"
    elif mode == "latest":
        sort_by = "submittedDate"
        sort_order = "descending"
    elif mode == "updated":
        sort_by = "lastUpdatedDate"
        sort_order = "descending"
    elif mode == "random":
        sort_by = "submittedDate"
        sort_order = "descending"
        start = random.randint(0, max(0, random_pool - max_results))
    else:
        raise ValueError("mode must be None, 'latest', 'updated', or 'random'")

    params = {
        "search_query": search_query,
        "start": start,
        "max_results": max_results if mode != "random" else random_pool,
        "sortBy": sort_by,
        "sortOrder": sort_order
    }

    url = BASE_URL + "?" + urllib.parse.urlencode(params)

    # SSL context
    context = ssl.create_default_context(cafile=certifi.where())
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "ArxivAgentTool/1.0 (your_email@example.com)"}
    )

    with urllib.request.urlopen(req, context=context) as response:
        data = response.read()

    # Namespaces
    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom"
    }

    root = ET.fromstring(data)
    papers = []

    for entry in root.findall("atom:entry", ns):
        arxiv_id = entry.find("atom:id", ns).text.split("/")[-1]
        title = entry.find("atom:title", ns).text.strip()
        summary = entry.find("atom:summary", ns).text.strip()
        published = entry.find("atom:published", ns).text.strip()
        updated = entry.find("atom:updated", ns).text.strip()

        # Authors
        authors = [
            author.find("atom:name", ns).text
            for author in entry.findall("atom:author", ns)
        ]

        # DOI
        doi_elem = entry.find("arxiv:doi", ns)
        doi = doi_elem.text if doi_elem is not None else None

        # Journal reference
        journal_elem = entry.find("arxiv:journal_ref", ns)
        journal_ref = journal_elem.text if journal_elem is not None else None

        # Comment
        comment_elem = entry.find("arxiv:comment", ns)
        comment = comment_elem.text if comment_elem is not None else None

        # Primary category
        primary_cat_elem = entry.find("arxiv:primary_category", ns)
        primary_category = primary_cat_elem.attrib["term"] if primary_cat_elem is not None else None

        # All categories
        categories = [cat.attrib["term"] for cat in entry.findall("atom:category", ns)]

        # Abstract URL
        abs_url = None
        for link in entry.findall("atom:link", ns):
            if link.attrib.get("rel") == "alternate":
                abs_url = link.attrib.get("href")
                break

        # PDF URL
        pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"

        papers.append({
            "id": arxiv_id,
            "title": title,
            "authors": authors,
            "published": datetime.fromisoformat(published.replace("Z", "")),
            "updated": datetime.fromisoformat(updated.replace("Z", "")),
            "abstract": summary,
            "pdf_url": pdf_url,
            "abs_url": abs_url,
            "doi": doi,
            "journal_ref": journal_ref,
            "comment": comment,
            "primary_category": primary_category,
            "all_categories": categories
        })

    # Final random sampling
    if mode == "random":
        papers = random.sample(papers, min(max_results, len(papers)))

    return papers


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

@tool(parse_docstring=True)
def arxiv_search(
    query: str,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
    max_results: int = 3
) -> Command:
    """Search arXiv for papers, store abstracts in Weaviate, retrieve top‑3, and save summaries to files.

    Args:
        query: The search query string.
        state: Injected agent state containing the current file system.
        tool_call_id: Injected tool call identifier for message responses.
        max_results

    Returns:
        Command updating the agent's files with saved paper abstracts and
        adding a tool message containing a summary of the retrieved papers.
    """
    # 1. Fetch papers from arXiv using custom function
    import time
    for attempt in range(3):          # up to 3 retries
        try:
            papers = get_arxiv_papers(query, max_results=max_results)
            break
        except Exception as e:
            if "429" in str(e) and attempt < 2:
                time.sleep(10 * (attempt + 1))  # 10s, then 20s
                continue
            return Command(
                update={
                    "messages": [
                        ToolMessage(f"arXiv search failed: {str(e)}", tool_call_id=tool_call_id)
                    ]
                }
            )

    if not papers:
        return Command(
            update={
                "messages": [
                    ToolMessage("No papers found on arXiv for this query.", tool_call_id=tool_call_id)
                ]
            }
        )

    # 2. Store in Weaviate (clear previous run)
    client = weaviate.connect_to_local(host="localhost", port=8080, grpc_port=50051)
    import time  # make sure this is at the top of your file

    try:
        if client.collections.exists("ArxivArticles"):
            client.collections.delete("ArxivArticles")
            for _ in range(10):
                if not client.collections.exists("ArxivArticles"):
                    break
                time.sleep(0.5)
    
        collection = client.collections.create(
            name="ArxivArticles",
            properties=[
                Property(name="title", data_type=DataType.TEXT),
                Property(name="abstract", data_type=DataType.TEXT),
                Property(name="arxiv_id", data_type=DataType.TEXT),
                Property(name="url", data_type=DataType.TEXT),
                Property(name="published", data_type=DataType.DATE),
            ],
            vectorizer_config=None
        )
        collection = client.collections.get("ArxivArticles")
    
        for paper in papers:
            title = paper["title"]
            abstract = paper["abstract"]
            text = title + " " + abstract
            embedding = embedding_model.encode(text).tolist()
            arxiv_id = paper["id"]
            url = paper.get("abs_url") or f"https://arxiv.org/abs/{arxiv_id}"
            
            published = paper["published"].isoformat()
            if not published.endswith('Z') and not ('+' in published or '-' in published[10:]):
                published += 'Z'
            
            collection.data.insert(
                properties={
                    "title": title,
                    "abstract": abstract,
                    "arxiv_id": arxiv_id,
                    "url": url,
                    "published": published,
                },
                vector=embedding
            )
    
        query_embedding = embedding_model.encode(query).tolist()
        response = collection.query.near_vector(
            near_vector=query_embedding,
            limit=3
        )
    finally:
        client.close()

    # 4. Save each abstract to a file and build a summary message
    files = state.get("files", {})
    saved_files = []
    summaries = []

    for i, obj in enumerate(response.objects, 1):
        props = obj.properties
        filename = f"arxiv_{props['arxiv_id']}_{uuid.uuid4().hex[:8]}.md"
        file_content = f"""# {props['title']}

**arXiv ID:** {props['arxiv_id']}
**URL:** {props['url']}
**Published:** {props['published']}

## Abstract
{props['abstract']}
"""
        files[filename] = file_content
        saved_files.append(filename)
        summaries.append(f"- {filename}: {props['title']}")

    summary_text = f"""📄 Found {len(papers)} papers on arXiv for '{query}'. Retrieved top 3 by relevance:

{chr(10).join(summaries)}

Files saved: {', '.join(saved_files)}

Use read_file() to read the full abstractsW when needed."""

    return Command(
        update={
            "files": files,
            "messages": [
                ToolMessage(summary_text, tool_call_id=tool_call_id)
            ],
        }
    )

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | Details
------------------------+------------+--------
embeddings.position_ids | UNEXPECTED |        

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Deep Agent

Now, we can just apply all of our prior learnings: 

* We'll give the researcher a `think_tool` and our `search_tool` above.
* We'll give our parent agent file tools, a `think_tool`, and a `task` tool. 

In [6]:
from datetime import datetime
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from utils import show_prompt, stream_agent

from file_tools_arxiv import ls, read_file, write_file
from prompts_arxiv import (
    FILE_USAGE_INSTRUCTIONS,
    RESEARCHER_INSTRUCTIONS,
    SUBAGENT_USAGE_INSTRUCTIONS,
    TODO_USAGE_INSTRUCTIONS,
)
from research_tools_arxiv import tavily_search, think_tool, get_today_str
from state_arxiv import DeepAgentState
from task_tool_arxiv import _create_task_tool
from todo_tools_arxiv import write_todos, read_todos

# -------------------- Main Agent Setup --------------------
model = init_chat_model(model="openai:gpt-4o-mini", temperature=0.0)

@tool   # no parse_docstring – uses type hints and plain docstring
def limited_arxiv_search(
    query: str,
    max_results: int,
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """Search arXiv for academic papers. Limited to 5 calls per research task."""
    if not hasattr(limited_arxiv_search.func, "call_count"):
        limited_arxiv_search.func.call_count = 0
    if limited_arxiv_search.func.call_count >= 5:
        limited_arxiv_search.func.call_count += 1   # optional, but better to increment even for error?
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        "ERROR: arXiv search limit reached (5 calls).",
                        tool_call_id=tool_call_id
                    )
                ]
            }
        )
    limited_arxiv_search.func.call_count += 1

    # Call the underlying function of the original tool.
    # Note: arxiv_search does NOT accept max_results; it always fetches 3.
    return arxiv_search.func(
        query=query,
        state=state,
        tool_call_id=tool_call_id,
        max_results=max_results
    )


# Tools for sub‑agent (the researcher)
sub_agent_tools = [limited_arxiv_search, tavily_search, think_tool]

# Built‑in tools for main agent
built_in_tools = [ls, read_file, write_file, write_todos, read_todos, think_tool]

# Research sub‑agent definition (uses string names that match the wrapped tools)
research_sub_agent = {
    "name": "research-agent",
    "description": "Delegate research to the sub-agent researcher. Only give this researcher one topic at a time.",
    "prompt": RESEARCHER_INSTRUCTIONS.format(date=get_today_str()),
    "tools": ["limited_arxiv_search", "tavily_search", "think_tool"],  # names must match actual tool names
}

# Create the task tool with our research sub‑agent
task_tool = _create_task_tool(
    sub_agent_tools,
    [research_sub_agent],
    model,
    DeepAgentState
)

all_tools = built_in_tools + [task_tool]

# Limits
max_concurrent_research_units = 2
max_researcher_iterations = 2
SUBAGENT_INSTRUCTIONS = SUBAGENT_USAGE_INSTRUCTIONS.format(
    max_concurrent_research_units=max_concurrent_research_units,
    max_researcher_iterations=max_researcher_iterations,
    date=datetime.now().strftime("%a %b %d, %Y").replace(" 0", " ")
)


In [7]:
show_prompt(SUBAGENT_INSTRUCTIONS)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│  You can delegate tasks to sub-agents.                                                                          │
│                                                                                                                 │
│  <Task>                                                                                                         │
│  Your role is to coordinate research by delegating specific research tasks to sub-agents.                       │
│  </Task>                                                                                                        │
│                                                                                                                 │
│  **Remember**: The sub‑agent has already gathered the information; now you need to read it and present it in a  │
│  user‑friendly way. Use `think_tool` if you need to reflect on how best to structure the answer.                │
│  <Available Tools>                                                                                              │
│  1. **task(description, subagent_type)**: Delegate research tasks to specialized sub-agents                     │
│     - description: Clear, specific research question or task                                                    │
│     - subagent_type: Type of agent to use (e.g., "research-agent")                                              │
│  2. **think_tool(reflection)**: Reflect on the results of each delegated task and plan next steps.              │
│     - reflection: Your detailed reflection on the results of the task and next steps.                           │
│                                                                                                                 │
│  **PARALLEL RESEARCH**: When you identify multiple independent research directions, make multiple **task**      │
│  tool calls in a single response to enable parallel execution. Use at most 2 parallel agents per iteration.     │
│  </Available Tools>                                                                                             │
│                                                                                                                 │
│  <Hard Limits>                                                                                                  │
│  **Task Delegation Budgets** (Prevent excessive delegation):                                                    │
│  - **Bias towards focused research** - Use single agent for simple questions, multiple only when clearly        │
│  beneficial or when you have multiple independent research directions based on the user's request.              │
│  - **Stop when adequate** - Don't over-research; stop when you have sufficient information                      │
│  - **Limit iterations** - Stop after 2 task delegations if you haven't found adequate sources                   │
│  </Hard Limits>                                                                                                 │
│                                                                                                                 │
│  <Scaling Rules>                                                                                                │
│  **Simple fact-finding, lists, and rankings** can use a single sub-agent:                                       │
│  - *Example*: "List the top 10 coffee shops in San Francisco" → Use 1 sub-agent, store in                       │
│  `findings_coffee_shops.md`                                                                                     │
│                                                                                                                 │
│  **Comparisons** can use a sub-agent for each element 

In [8]:
INSTRUCTIONS = (
    "# TODO MANAGEMENT\n"
    + TODO_USAGE_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# FILE SYSTEM USAGE\n"
    + FILE_USAGE_INSTRUCTIONS
    +"When you receive the researcher’s final answer, always include the list of sources they provided."
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + "# SUB-AGENT DELEGATION\n"
    + "Make sure to call sub-agent atleast once - in other words call think tool atleast once. \n"
    + SUBAGENT_INSTRUCTIONS
)

show_prompt(INSTRUCTIONS)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│  # TODO MANAGEMENT                                                                                              │
│  Based upon the user's request:                                                                                 │
│  1. Use the write_todos tool to create TODO at the start of a user request, per the tool description.           │
│  2. After you accomplish a TODO, use the read_todos to read the TODOs in order to remind yourself of the plan.  │
│  3. Reflect on what you've done and the TODO.                                                                   │
│  4. Mark you task as completed, and proceed to the next TODO.                                                   │
│  5. Continue this process until you have completed all TODOs.                                                   │
│                                                                                                                 │
│  IMPORTANT: Always create a research plan of TODOs and conduct research following the above guidelines for ANY  │
│  user request.                                                                                                  │
│  IMPORTANT: Aim to batch research tasks into a *single TODO* in order to minimize the number of TODOs you have  │
│  to keep track of.                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
│  ================================================================================                               │
│                                                                                                                 │
│  # FILE SYSTEM USAGE                                                                                            │
│  You have access to a virtual file system to help you retain and save context.                                  │
│                                                                                                                 │
│  ## Workflow Process                                                                                            │
│  1. **Orient**: Use ls() to see existing files before starting work                                             │
│  2. **Save**: Use write_file() to store the user's request so that we can keep it for later                     │
│  3. **Research**: Proceed with research. The search tool will write files.                                      │
│  4. **Read**: Once you are satisfied with the collected sources, read the files and use them to answer the      │
│  user's question directly.                                                                                      │
│  When you receive the researcher’s final answer, always include the list of sources they provided.              │
│                                                                                                                 │
│  ================================================================================                               │
│                                                                                                                 │
│  # SUB-AGENT DELEGATION                                                                                         │
│  Make sure to call sub-agent atleast once - in other words call think tool atleast once.                        │
│  You can delegate tasks to sub-agents.                                                                          │
│                                                       

In [9]:
# Create the main agent
main_agent = create_agent(
    model,
    all_tools,
    system_prompt=INSTRUCTIONS,
    state_schema=DeepAgentState
)

# -------------------- Helper for Random Fact --------------------
# Simple year→event mapping (can be expanded)
YEAR_EVENTS = {
    2020: "the COVID‑19 pandemic began",
    2021: "NASA's Perseverance rover landed on Mars",
    2022: "the James Webb Space Telescope released its first images",
    2023: "the AI boom continued with GPT‑4",
    2024: "the Paris Olympics were held",
}

def get_random_fact(year: int) -> str:
    if year in YEAR_EVENTS:
        return f"Fun fact: This paper was published in {year}, the year {YEAR_EVENTS[year]}."
    else:
        return f"Fun fact: This paper was published in {year}."

In [10]:
show_prompt(RESEARCHER_INSTRUCTIONS)

╭──────────────────────────────────────────────────── Prompt ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│  You are a research assistant conducting research on the user's input topic. For context, today's date is       │
│  {date}.                                                                                                        │
│                                                                                                                 │
│  <Task>                                                                                                         │
│  Your job is to use tools to gather information about the user's input topic.                                   │
│  You can use any of the tools provided to you to find resources that can help answer the research question.     │
│  You can call these tools in series or in parallel, your research is conducted in a tool-calling loop.          │
│  </Task>                                                                                                        │
│                                                                                                                 │
│  <Available Tools>                                                                                              │
│  You have access to two main tools:                                                                             │
│  1. **tavily_search** – Use this to find factual information, explanations, and up‑to‑date content on the web.  │
│     *This is your primary tool for answering the user's question.*                                              │
│     *You can make only one call of this function*                                                               │
│                                                                                                                 │
│  2. **limited_arxiv_search** – Use this **only** to find relevant academic papers and their PDF links.          │
│     *Do not attempt to answer the question using arXiv abstracts – they are too brief. Instead, collect paper   │
│  titles, arXiv IDs, and PDF URLs to include as references in your final answer.*                                │
│                                                                                                                 │
│  3. **think_tool** – Use this after each search to reflect on findings and plan next steps.                     │
│                                                                                                                 │
│  **IMPORTANT GUIDELINES FOR LIMITED_ARXIV_SEARCH:**                                                             │
│  - arXiv works best with **simple keyword queries - maximum 3 words can be provided** (e.g., "few-shot          │
│  learning", "transformer architecture", "attention mechanism").                                                 │
│  - **DO NOT** use full sentences or questions like "Explain few-shot learning in large language models" – this  │
│  will likely return no results.                                                                                 │
│  - If your initial query returns no papers, try:                                                                │
│    - Using more general keywords (e.g., "meta-learning" instead of "few-shot learning in LLMs")                 │
│    - Removing adjectives and focusing on core terms                                                             │
│    - Checking spelling and using alternative terminology                                                        │
│                                                                                                                 │
│  **CRITICAL: Use think_tool after each search to reflect on results and plan next steps**                       │
│  </Available Tools>                                   

In [11]:
# We'll add the random fact after the agent finishes, by processing the final message.
# Alternatively, we could add a final step in the agent, but for simplicity we'll append it after invocation.

initial_state = {
    "messages": [{"role": "user", "content": "Supervised Learning"}],
    "todos": [],
    "files": {}
}
result = main_agent.invoke(initial_state, config={"configurable": {"thread_id": "1"}})

# Extract final answer (last message from assistant)
final_answer = result["messages"][-1].content

# Try to extract a year from any file that might contain a publication date
# For simplicity, we'll just pick a random fact from the current year if no paper found
# In a real implementation, you might parse the saved files.
import random
fact_year = random.choice(list(YEAR_EVENTS.keys()))  # just for demonstration
fact = get_random_fact(fact_year)

print("\n" + "="*50)
print("FINAL ANSWER:")
print(final_answer)
print("\n" + fact)
print("="*50)

C:\projekty_local\deep-agents-from-scratch\.venv\Lib\site-packages\pydantic\v1\main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)
C:\projekty_local\deep-agents-from-scratch\.venv\Lib\site-packages\bs4\element.py:1487: ResourceWarning: unclosed <ssl.SSLSocket fd=2384, family=2, type=1, proto=0, laddr=('192.168.1.10', 58847), raddr=('23.192.90.236', 443)>
  u = str.__new__(cls, value)



FINAL ANSWER:
### Supervised Learning Overview

**Definition:**
Supervised learning is a type of machine learning where an algorithm learns to map input data to specific outputs based on example input-output pairs. This process involves training a statistical model using labeled data, meaning each piece of input data is associated with the correct output. The primary goal is for the trained model to accurately predict outputs for new, unseen data.

**Types of Supervised Learning:**
1. **Classification:** Predicting a categorical label (e.g., spam detection).
2. **Regression:** Predicting a continuous value (e.g., house prices).

**Common Algorithms:**
- **Linear Regression:** For regression tasks.
- **Logistic Regression:** For binary classification.
- **Decision Trees:** Versatile for both classification and regression.
- **Support Vector Machines (SVM):** Effective for high-dimensional spaces.
- **Neural Networks:** Useful for complex tasks like image and speech recognition.

**Appl

In [ ]:
BASE_URL = "https://export.arxiv.org/api/query"


def get_arxiv_papers(
    query,
    max_results=10,
    start=0,
    mode=None,          # None → default relevance
    random_pool=100
):
    """
    Retrieve arXiv papers with extended metadata.

    Parameters:
        query (str): search keywords
        max_results (int): number of results to return
        start (int): pagination offset
        mode (str|None): 'latest', 'updated', 'random', or None (relevance)
        random_pool (int): pool size for random mode (used only for random)

    Returns:
        List[dict]: each dict contains paper metadata
    """

    if not query:
        raise ValueError("query must not be empty")

    search_query = f'all:"{query}"'

    # -----------------------
    # Sorting logic
    # -----------------------
    if mode is None:
        sort_by = "relevance"
        sort_order = "descending"
    elif mode == "latest":
        sort_by = "submittedDate"
        sort_order = "descending"
    elif mode == "updated":
        sort_by = "lastUpdatedDate"
        sort_order = "descending"
    elif mode == "random":
        sort_by = "submittedDate"
        sort_order = "descending"
        start = random.randint(0, max(0, random_pool - max_results))
    else:
        raise ValueError("mode must be None, 'latest', 'updated', or 'random'")

    params = {
        "search_query": search_query,
        "start": start,
        "max_results": max_results if mode != "random" else random_pool,
        "sortBy": sort_by,
        "sortOrder": sort_order
    }

    url = BASE_URL + "?" + urllib.parse.urlencode(params)

    # SSL context
    context = ssl.create_default_context(cafile=certifi.where())
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "ArxivAgentTool/1.0 (your_email@example.com)"}
    )

    with urllib.request.urlopen(req, context=context) as response:
        data = response.read()

    # Namespaces
    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom"
    }

    root = ET.fromstring(data)
    papers = []

    for entry in root.findall("atom:entry", ns):
        arxiv_id = entry.find("atom:id", ns).text.split("/")[-1]
        title = entry.find("atom:title", ns).text.strip()
        summary = entry.find("atom:summary", ns).text.strip()
        published = entry.find("atom:published", ns).text.strip()
        updated = entry.find("atom:updated", ns).text.strip()

        # Authors
        authors = [
            author.find("atom:name", ns).text
            for author in entry.findall("atom:author", ns)
        ]

        # DOI
        doi_elem = entry.find("arxiv:doi", ns)
        doi = doi_elem.text if doi_elem is not None else None

        # Journal reference
        journal_elem = entry.find("arxiv:journal_ref", ns)
        journal_ref = journal_elem.text if journal_elem is not None else None

        # Comment
        comment_elem = entry.find("arxiv:comment", ns)
        comment = comment_elem.text if comment_elem is not None else None

        # Primary category
        primary_cat_elem = entry.find("arxiv:primary_category", ns)
        primary_category = primary_cat_elem.attrib["term"] if primary_cat_elem is not None else None

        # All categories
        categories = [cat.attrib["term"] for cat in entry.findall("atom:category", ns)]

        # Abstract URL
        abs_url = None
        for link in entry.findall("atom:link", ns):
            if link.attrib.get("rel") == "alternate":
                abs_url = link.attrib.get("href")
                break

        # PDF URL
        pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"

        papers.append({
            "id": arxiv_id,
            "title": title,
            "authors": authors,
            "published": datetime.fromisoformat(published.replace("Z", "")),
            "updated": datetime.fromisoformat(updated.replace("Z", "")),
            "abstract": summary,
            "pdf_url": pdf_url,
            "abs_url": abs_url,
            "doi": doi,
            "journal_ref": journal_ref,
            "comment": comment,
            "primary_category": primary_category,
            "all_categories": categories
        })

    # Final random sampling
    if mode == "random":
        papers = random.sample(papers, min(max_results, len(papers)))

    return papers

In [ ]:
import arxiv

# Construct the default API client.
client = arxiv.Client()

# Search for the 10 most recent articles matching the keyword "quantum."
search = arxiv.Search(
  query = "machine learning random forest",
  max_results = 2
)

# `results` is a generator; you can iterate over its elements one by one...
for r in client.results(search):
  print(r)

In [25]:
# 3. Retrieve top‑3 most relevant abstracts using query embedding
query_embedding = embedding_model.encode('Explain how large language models works. Return references to found papers and used web pages').tolist()
response = collection.query.near_vector(
    near_vector=query_embedding,
    limit=3
)

NameError: name 'collection' is not defined

In [ ]:
from datetime import datetime
from sentence_transformers import SentenceTransformer
from IPython.display import Image, display
from langchain.chat_models import init_chat_model
#from langgraph.prebuilt import create_react_agent
from langchain.agents import create_agent
from utils import show_prompt, stream_agent

from file_tools_arxiv import ls, read_file, write_file
from prompts_arxiv import (
    FILE_USAGE_INSTRUCTIONS,
    RESEARCHER_INSTRUCTIONS,
    SUBAGENT_USAGE_INSTRUCTIONS,
    TODO_USAGE_INSTRUCTIONS,
)
from research_tools_arxiv import tavily_search, think_tool, get_today_str
from state_arxiv import DeepAgentState
from task_tool_arxiv import _create_task_tool
from todo_tools_arxiv import write_todos, read_todos

import weaviate
import weaviate.classes as wvc
from sentence_transformers import SentenceTransformer
from langchain_core.messages import ToolMessage, HumanMessage
from langchain_core.tools import InjectedToolCallId, tool
from langchain_core.documents import Document
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import TavilySearchResults
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.prebuilt import InjectedState
from langgraph.types import Command
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
from markdownify import markdownify
from tavily import TavilyClient
from langchain_core.messages import AnyMessage, BaseMessage
from langgraph.graph.message import add_messages
BASE_URL = "https://export.arxiv.org/api/query"

def get_arxiv_papers(
    query,
    max_results=3,
    start=0,
    mode=None,          # None → default relevance
    random_pool=100
):
    """
    Retrieve arXiv papers with extended metadata.

    Parameters:
        query (str): search keywords
        max_results (int): number of results to return
        start (int): pagination offset
        mode (str|None): 'latest', 'updated', 'random', or None (relevance)
        random_pool (int): pool size for random mode (used only for random)

    Returns:
        List[dict]: each dict contains paper metadata
    """
    if not query:
        raise ValueError("query must not be empty")

    search_query = query

    # Sorting logic
    if mode is None:
        sort_by = "relevance"
        sort_order = "descending"
    elif mode == "latest":
        sort_by = "submittedDate"
        sort_order = "descending"
    elif mode == "updated":
        sort_by = "lastUpdatedDate"
        sort_order = "descending"
    elif mode == "random":
        sort_by = "submittedDate"
        sort_order = "descending"
        start = random.randint(0, max(0, random_pool - max_results))
    else:
        raise ValueError("mode must be None, 'latest', 'updated', or 'random'")

    params = {
        "search_query": search_query,
        "start": start,
        "max_results": max_results if mode != "random" else random_pool,
        "sortBy": sort_by,
        "sortOrder": sort_order
    }

    url = BASE_URL + "?" + urllib.parse.urlencode(params)

    # SSL context
    context = ssl.create_default_context(cafile=certifi.where())
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "ArxivAgentTool/1.0 (your_email@example.com)"}
    )

    with urllib.request.urlopen(req, context=context) as response:
        data = response.read()

    # Namespaces
    ns = {
        "atom": "http://www.w3.org/2005/Atom",
        "arxiv": "http://arxiv.org/schemas/atom"
    }

    root = ET.fromstring(data)
    papers = []

    for entry in root.findall("atom:entry", ns):
        arxiv_id = entry.find("atom:id", ns).text.split("/")[-1]
        title = entry.find("atom:title", ns).text.strip()
        summary = entry.find("atom:summary", ns).text.strip()
        published = entry.find("atom:published", ns).text.strip()
        updated = entry.find("atom:updated", ns).text.strip()

        # Authors
        authors = [
            author.find("atom:name", ns).text
            for author in entry.findall("atom:author", ns)
        ]

        # DOI
        doi_elem = entry.find("arxiv:doi", ns)
        doi = doi_elem.text if doi_elem is not None else None

        # Journal reference
        journal_elem = entry.find("arxiv:journal_ref", ns)
        journal_ref = journal_elem.text if journal_elem is not None else None

        # Comment
        comment_elem = entry.find("arxiv:comment", ns)
        comment = comment_elem.text if comment_elem is not None else None

        # Primary category
        primary_cat_elem = entry.find("arxiv:primary_category", ns)
        primary_category = primary_cat_elem.attrib["term"] if primary_cat_elem is not None else None

        # All categories
        categories = [cat.attrib["term"] for cat in entry.findall("atom:category", ns)]

        # Abstract URL
        abs_url = None
        for link in entry.findall("atom:link", ns):
            if link.attrib.get("rel") == "alternate":
                abs_url = link.attrib.get("href")
                break

        # PDF URL
        pdf_url = f"https://arxiv.org/pdf/{arxiv_id}.pdf"

        papers.append({
            "id": arxiv_id,
            "title": title,
            "authors": authors,
            "published": datetime.fromisoformat(published.replace("Z", "")),
            "updated": datetime.fromisoformat(updated.replace("Z", "")),
            "abstract": summary,
            "pdf_url": pdf_url,
            "abs_url": abs_url,
            "doi": doi,
            "journal_ref": journal_ref,
            "comment": comment,
            "primary_category": primary_category,
            "all_categories": categories
        })

    # Final random sampling
    if mode == "random":
        papers = random.sample(papers, min(max_results, len(papers)))

    return papers

# BBVCcv